Question 1

In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [2]:
documents[:3]

[{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a sim

In [3]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [4]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

from dotenv import load_dotenv
from google import genai

load_dotenv()
gemini_client = genai.Client()

In [5]:
import json
user_prompt = json.dumps(documents[1])

In [6]:
from evaluation_utils import llm_structured_gemini_retry

out, usage = llm_structured_gemini_retry(
    gemini_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
out

Questions(questions=['Are there any specific tools or software I need to install before I start coding?', 'What is the recommended approach for managing my API keys so they do not accidentally get shared publicly?', 'How do I set up a new project environment using the uv package manager?', 'Is it possible to use alternative services besides OpenAI for this course?', 'What is the easiest way to avoid manually loading my environment variables in every notebook I create?'])

In [8]:
usage

GenerateContentResponseUsageMetadata(
  candidates_token_count=108,
  prompt_token_count=1400,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=1400
    ),
  ],
  total_token_count=1508
)

Question 2

In [9]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [2]:
from embedder import Embedder

embedder = Embedder()

import numpy as np

X = np.array([embedder.encode(c["content"]) for c in chunks])
X.shape

ModuleNotFoundError: No module named 'onnxruntime'

In [ ]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["filename"])
vindex.fit(X, chunks)

In [ ]:
from rag_helper import build_index

index = build_index(documents)

In [ ]:
def hybrid_search(query, k=60):
    text_results = index.search(query, num_results=10)
    vector_results = vindex.search(embedder.encode(query), num_results=10)
    return rrf([text_results, vector_results], k=k)

In [22]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [15]:
import json
from evaluation_utils import llm_structured_gemini_retry

def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_gemini_retry(
        gemini_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            #"document": doc["filename"]
        })

    return results, usage

In [16]:
generate_ground_truth(documents[:3])

([{'question': 'Why do large language models sometimes give incorrect answers even when they sound like they know what they are talking about?'},
  {'question': 'If I am building an application, how can I make sure the model has access to my own private documents or specific company information?'},
  {'question': 'Do I really need to install and learn a bunch of complex frameworks, or can I build a functional system just using basic Python code?'},
  {'question': 'What are the specific components that make up a basic Retrieval-Augmented Generation pipeline?'},
  {'question': 'Why is it considered a security risk to include my API key directly inside my project files before uploading them to a repository?'}],
 GenerateContentResponseUsageMetadata(
   candidates_token_count=143,
   prompt_token_count=4089,
   prompt_tokens_details=[
     ModalityTokenCount(
       modality=<MediaModality.TEXT: 'TEXT'>,
       token_count=4089
     ),
   ],
   total_token_count=4232
 ))